### Getting Started:
- Make sure you are outside the sam2 repository, then run `pip install -r requirements.txt`

In [1]:
import torch
import numpy as np
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
import matplotlib.pyplot as plt
import cv2
import pandas as pd

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

In [3]:
def show_anns(anns, borders=True, path='example_image.png'):
    """
    NOTE: This function assumes you are saving to a folder called 'output_masks' 
    in the parent dir.
    """
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)
 
    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    # print("Initial canvas:", np.array(img))
    
    # Set default pixel transparency to ... (0 for transparent, 1 for full)
    img[:,:,3] = 1 

    # At this point, sorted_anns is a list of the segmented masks SAM2 found in the data
    for ann in sorted_anns:
        # For every area in the segmentation, get the mask at anns['segmentation']...
        m = ann['segmentation']
        
        # Use a color (that is NOT white)
        color_mask = np.concatenate([np.random.uniform(low=0.1, high=1, size=(3)), [1]])
        img[m] = color_mask 
        # and plot the colors
        if borders:
            contours, _ = cv2.findContours(m.astype(np.uint8),cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
            # Try to smooth contours
            contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
            cv2.drawContours(img, contours, -1, (0,0,1,0.4), thickness=1) 
 

    ax.imshow(img)
    # print("Anns mask", sorted_anns[0]['segmentation'].shape)
    # print("Image as array:", np.array(img))

    

    plt.imsave(f"../output_masks/{path}", img)

In [4]:
def noisify_image(image:np.array, noise:str="random", threshold=0.1):
    # add noise to image
    # snow is generally greyscale, so noise values should be in that range
        # uint8 dtype limits cv2 randn to [0, 255]
    noise_mask = np.zeros(shape=(image.shape[0], image.shape[1], 3), dtype=np.uint8)

    if noise == "gaussian":
        # apply gaussian noise
        cv2.randn(noise_mask, mean=(128, 128, 128), stddev=(40, 40, 40))
        noise_mask = (noise_mask * 0.5).astype(np.uint8) # dilute the noise so that its application to the iamge is more realistic
        image = np.add(image, noise_mask)

    # add more noises methods here...
    elif noise == "random":
        for i in range(image.shape[0]):
            for j in range(image.shape[1]):
                if np.random.random() <= threshold:
                    image[i][j] = (np.random.rand(3) * 255).astype(np.uint32)
        

    return image

In [5]:
checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
mask_generator = SAM2AutomaticMaskGenerator(build_sam2(model_cfg, checkpoint, device=device, apply_postprocessing=False))

In [ ]:
image=Image.open("../0001TP_009240.png")
image=np.array(image.convert("RGB"))

print("Before Noise:")
plt.figure(figsize=(20, 20))
plt.imshow(image)
plt.axis('off')
plt.show()

# thresholds 0.3 and above are a bit brutal - like a snow day snowstorm.
image = noisify_image(image, noise="random", threshold=0.1)

print("After Noise:")
plt.figure(figsize=(20, 20))
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
masks = mask_generator.generate(image)

plt.figure(figsize=(20,20))
plt.imshow(image)
show_anns(masks)
plt.axis('off')
plt.show() 

In [ ]:
# Getting individual masks from SAM2

# Example image - clean, no noise
image=Image.open("../0001TP_009240.png")
image=np.array(image.convert("RGB"))

masks = mask_generator.generate(image)
for key in masks[0]: # assume sam2 has 1 or more masks
    print(key, ":", masks[0][key])

print("--------------------------------------------")
print("SAM2 Output Mask example shape:", masks[0]['segmentation'].shape)
print("Input image shape:", image.shape)

print("--------------------------------------------")
first_component = image
for i in range(first_component.shape[0]):
    for j in range(first_component.shape[1]):
        if not masks[0]['segmentation'][i][j]:
            first_component[i][j] = [0,0,0] 

plt.figure(figsize=(20, 20))
plt.imshow(first_component)
plt.axis('off')
plt.show()

## Common Bugs:
- "Torch is not compiled with Cuda" - uninstall torch, torchvision, torchaudio (`pip uninstall torch torchvision torchaudio`) and install the newest versions of each w/ Cuda (command available on Pytorch website). 
    - ATM this is:
     ```pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124```
- "Not implemented error" - this issue occurs when the device is Cuda but the Cuda version is not compatible with the GPU. Again, same fix as above (essentially update your Cuda version)
- "Commit denied by prereceive hook" - this issue occurs when the notebook is run & committed with all output. The output images in the notebook cause it to grow beyond git's acceptable size (100-120 mb). To fix, clear all output and restart kernel before committing.

In [9]:
import os
import random

dataset_path = "../download_dataset"
output_path = "../output_masks/val"

val_dir = os.path.join(dataset_path, 'val')

In [10]:
k = 0 # to see effects, set k > 0
val_files = os.listdir(val_dir)
noise="random" # Disable by setting to None
show_img = True

for file in random.choices(val_files, k=k):
    try:
        src = os.path.join(val_dir, file)
        
        # Get original image
        image=Image.open(src)
        image=np.array(image.convert("RGB"))
        
        if show_img:
            plt.figure(figsize=(20, 20))
            plt.imshow(image)
            plt.axis('off')
            plt.show()
            print("Original Image")

        if noise:
            image = noisify_image(image)
            
            if show_img:
                print("After Noise:")
                plt.figure(figsize=(20, 20))
                plt.imshow(image)
                plt.axis('off')
                plt.show()

        # Show SAM2 Output
        masks = mask_generator.generate(image)
        plt.figure(figsize=(20,20))
        plt.imshow(image)
        show_anns(masks, path=f"val/{file}")
        plt.axis('off')
        plt.show() 
        print("SAM2 Segmented Output")

        # Show Ground Truth
        file_labelled = file[:-4] + "_L" + file[-4:] # labelled files have a '_L' in them
        src_labelled = os.path.join(dataset_path, 'val_labels')
        src_labelled = os.path.join(src_labelled, file_labelled) # Get the related labeled image filepath
        image=Image.open(src_labelled)
        image=np.array(image.convert("RGB"))
        plt.figure(figsize=(20, 20))
        plt.imshow(image)
        plt.axis('off')
        plt.show()
        print("Ground Truth")
    except:
        print("An image could not be found. Moving on...")

In [ ]:
# Testing SAM2's resilience to noise
max_times_ten=0 # set to 5 to see full range

found_image = False
while not found_image:
    try:
        file = random.choice(val_files)
        src = os.path.join(val_dir, file)    
        # Get original image
        image=Image.open(src)
        image=np.array(image.convert("RGB"))
        found_image = True
    except:
        print("Could not find an image. Trying another ...")

# Test SAM2's resilience to noise
for i in range(1, max_times_ten+1):
    t = i / 10
    image = noisify_image(image, threshold=t)
    
    print(f"Noisy Image at threshold {i / 10}:")
    plt.figure(figsize=(20, 20))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

    # Show SAM2 Output
    masks = mask_generator.generate(image)
    plt.figure(figsize=(20,20))
    plt.imshow(image)
    show_anns(masks, path=f"val/threshold_{t}_{file}")
    plt.axis('off')
    plt.show() 
    print("SAM2 Segmented Output")

# Show Ground Truth
try:
    file_labelled = file[:-4] + "_L" + file[-4:] # labelled files have a '_L' in them
    src_labelled = os.path.join(dataset_path, 'val_labels')
    src_labelled = os.path.join(src_labelled, file_labelled) # Get the related labeled image filepath
    image=Image.open(src_labelled)
    image=np.array(image.convert("RGB"))
    plt.figure(figsize=(20, 20))
    plt.imshow(image)
    plt.axis('off')
    plt.show()
    print("Ground Truth")
except:
    print("Could not find ground truth. Moving on...")

<h2>YOLO & SAM2</h2>

In [ ]:
%pip install ultralytics

In [ ]:
table = pd.read_csv("../download_dataset/class_dict.csv")
# print(table.head())

# dict based on english labels
class_dict = {row["name"].lower() : [row["r"] / 255, row["g"] / 255, row["b"] / 255] for _, row in table.iterrows()}
color_labels = [class_dict[key] for key in class_dict]

print("Class dict:")
print(class_dict)
label_map = {
    'person':'pedestrian'
}

def label_to_color(label:str):
    if label in class_dict:
            color = np.concatenate([np.array(class_dict[label]), np.array([1])], axis=0)
    else:
        if label in label_map:
            label = label_map[label]
            if label in class_dict:
                color = np.concatenate([np.array(class_dict[label]), np.array([1])], axis=0)
            else:
                print("Could not find", label)
                color = np.concatenate([np.random.random(3), np.array([1])], axis=0)
        else:
            print("Could not find", label)
            color = np.concatenate([np.random.random(3), np.array([1])], axis=0)
    return color

In [14]:
def show_mask(mask, ax, label=None, index=-1):
    # edit so that color aligns w/ camvid class dict

    if label:
        color = label_to_color(label)
    else:
        if index > -1 and index < len(color_labels):
            color = np.concatenate([np.array(color_labels[index]), np.array([1])], axis=0)
        else:
            color = np.concatenate([np.random.random(3), np.array([1])], axis=0)
    
    
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)
    
def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   
    
def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0,0,0,0), lw=2))    

### MASKS ARE 720 * 960 boolean masks of the image

In [15]:
# bbox = np.reshape(bbox, shape=(-1,1)).shape
from typing import List

def SAM2_segment_box(image, predictor, bbox, labels=None)->List:
    predictor.set_image(image=image)
    plt.figure(figsize=(10, 10))
    plt.imshow(image)

    n = len(bbox)
    # print(n)

    object_mask_dict = {}
    list_of_masks = [] # final length should match n
    for i in range(n):
        obj_detected, label = bbox[i], labels[i]

        input_box = np.array(obj_detected)

        masks, _, _ = predictor.predict(
            point_coords=None,
            point_labels=None,
            box=obj_detected,
            multimask_output=False
        )
        
        # print("Mask format:")
        # print(np.shape(masks[0])) # 720 X  960
        # print(np.max(masks[0])) # 1.0
        show_mask(masks[0], plt.gca(), label=label)
        show_box(input_box, plt.gca())

        if label in object_mask_dict:
            # union the two masks
            object_mask_dict[label] = np.logical_or(masks[0], object_mask_dict[label])
        else:
            object_mask_dict[label] = masks[0]
        list_of_masks.append(masks[0])
    plt.axis('off')
    plt.show()
    return object_mask_dict, list_of_masks

In [16]:
from ultralytics import YOLO

img_path = "../0001TP_009240.png"

yolo_model = YOLO('yolov8n.pt')

sam2_checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
predictor = SAM2ImagePredictor(sam2_model)

In [ ]:
results = yolo_model.predict(source=img_path, conf=0.25)
names = results[0].names

for result in results: # Usually results is of length 1 - can change
    boxes = result.boxes
    bbox = boxes.xyxy.tolist() # bounding boxes AKA areas where YOLO found an object
    
    # print(boxes.cls)
    # print(result.names)

    image=Image.open(img_path)
    image=np.array(image.convert("RGB"))

    plt.figure(figsize=(20, 20))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

    object_mask_dict, list_of_masks = SAM2_segment_box(image=image, predictor=predictor, bbox=bbox, labels = [names[index] for index in boxes.cls.tolist()])
    print(f"Found {len(list_of_masks)} objects in image.") 

In [ ]:
# Verify that we work
print(image.shape[1])
for label in object_mask_dict:
    mask_overall = object_mask_dict[label]
    color = label_to_color(label)[:-1]
    canvas = np.ones((720, 960, 3), dtype=np.float32)
    
    # print(mask_overall.shape)
    # print(canvas.shape)


    for i in range(mask_overall.shape[0]):
        for j in range(mask_overall.shape[1]):
            if mask_overall[i][j]:
                # print(canvas[i][j])
                # print(color)
                canvas[i][j] = color

    plt.figure(figsize=(10, 10))
    plt.imshow(canvas)    
    plt.axis('off')
    plt.show()

<h1>Fine-Tuning SAM2 w/ CamVid</h1>

In [7]:
import os
import numpy as np  # For debugging prints.
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

In [9]:
# ------------------------------
# 1. Custom Dataset for Image/Mask Pairs
# ------------------------------
class CamVidDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform_image=None, transform_mask=None, debug=False):
        """
        Args:
            image_dir (str): Directory containing input images.
            mask_dir (str): Directory containing ground truth segmentation masks.
            transform_image (callable, optional): Transformations to apply to the image.
            transform_mask (callable, optional): Transformations to apply to the mask.
            debug (bool): If True, print debugging information.
        """
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        # Only include raw images (those without "_L" in the filename)
        self.image_files = sorted([f for f in os.listdir(image_dir)
                                   if f.endswith('.png') and "_L" not in f])
        self.transform_image = transform_image
        self.transform_mask = transform_mask
        self.debug = debug

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        image_filename = self.image_files[index]
        # Construct the mask filename by appending "_L" before the extension.
        base, ext = os.path.splitext(image_filename)
        mask_filename = base + "_L" + ext

        image_path = os.path.join(self.image_dir, image_filename)
        mask_path = os.path.join(self.mask_dir, mask_filename)

        # Open image and mask.
        # Debug: print raw image shape (may include an alpha channel)
        raw_image = Image.open(image_path)
        if self.debug:
            print(f"Raw image shape for {image_filename}: {np.array(raw_image).shape}")

        # Convert image to RGB (ensures 3 channels)
        image = raw_image.convert("RGB")
        if self.debug:
            print(f"Converted image shape for {image_filename}: {np.array(image).shape}")

        # Open mask in grayscale.
        mask = Image.open(mask_path).convert("L")
        if self.debug:
            print(f"Mask shape for {mask_filename}: {np.array(mask).shape}")

        if self.transform_image:
            image = self.transform_image(image)
        if self.transform_mask:
            mask = self.transform_mask(mask)

        return image, mask

In [ ]:
# Set desired image size (height, width)
IMAGE_SIZE = (1024, 1024)

# Define transformations
image_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
])

mask_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
])

# Update these paths as needed
IMAGE_DIR = "../download_dataset/train"
MASK_DIR = "../download_dataset/train_labels"

# Create dataset and dataloader
dataset = CamVidDataset(IMAGE_DIR, MASK_DIR,
                        transform_image=image_transform,
                        transform_mask=mask_transform,
                        debug=True)

# Reduce batch size to 1 to debug dimension issues
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=0) # assumes 4 threads - works if 0

# Load SAM2 model
sam2_checkpoint = "./checkpoints/sam2.1_hiera_small.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
# print(sam2_model) # to see attributes

# Freeze encoders
for param in sam2_model.image_encoder.parameters():
    param.requires_grad = False
for param in sam2_model.sam_prompt_encoder.parameters():
    param.requires_grad = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sam2_model.to(device)

optimizer = optim.Adam(sam2_model.sam_mask_decoder.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()

num_epochs = 10
for epoch in range(num_epochs):
    sam2_model.train()
    for batch_idx, (images, gt_masks) in enumerate(dataloader):
        images = images.to(device)
        gt_masks = gt_masks.to(device)

        # Get image embeddings
        with torch.no_grad():
            image_embeddings = sam2_model.image_encoder(images)

        print("image embeddings:", image_embeddings)

        # Create prompt boxes
        batch_size = images.size(0)
        W, H = IMAGE_SIZE[1], IMAGE_SIZE[0]
        boxes = torch.tensor([[0, 0, W, H]] * batch_size, device=device)

        # Get prompt embeddings
        with torch.no_grad():
            sparse_embeddings, dense_embeddings = sam2_model.sam_prompt_encoder(
                points=None,
                boxes=boxes,
                masks=None,
            )

        # Debug print shapes
        if batch_idx == 0:
            print(f"Batch size: {batch_size}")
            print(f"Image embeddings shape: {image_embeddings.shape}")
            print(f"Dense embeddings shape: {dense_embeddings.shape}")
            print(f"Sparse embeddings shape: {sparse_embeddings.shape}")

        # Generate masks
        low_res_masks, iou_predictions = sam2_model.sam_mask_decoder(
            image_embeddings=image_embeddings,
            image_pe=sam2_model.sam_prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=False,
        )

        # Post-process masks
        upscaled_masks = sam2_model.postprocess_masks(
            low_res_masks,
            input_size=IMAGE_SIZE,
            original_size=IMAGE_SIZE
        ).to(device)

        # Convert to binary mask
        binary_mask = torch.sigmoid(upscaled_masks)

        # Compute loss
        loss = loss_fn(binary_mask, gt_masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch [{epoch + 1}/{num_epochs}], "
                f"Batch [{batch_idx + 1}/{len(dataloader)}], "
                f"Loss: {loss.item():.4f}")

        # Optional: Save intermediate results periodically
        if (batch_idx + 1) % 100 == 0:
            torch.save(sam2_model.state_dict(), f"sam_checkpoint_epoch{epoch}_batch{batch_idx}.pth")

# Save final model
torch.save(sam2_model.state_dict(), "fine_tuned_sam2.pth")
print("Training complete. Model saved as 'fine_tuned_sam2.pth'.")

### No YOLOv2
[Helpful Link](https://medium.com/towards-data-science/train-fine-tune-segment-anything-2-sam-2-in-60-lines-of-code-928dd29a63b3)

In [ ]:
# train_dir = "../download_dataset/train/" # Path to train folder 
# data_dir = "../download_dataset/train_labels/" # Path to truth data folder - sample from it for training
# data = []

# for ff, name in enumerate(os.listdir(data_dir)):
#     data.append({
#         "image":train_dir + name[:-6] + ".png",
#         "annotated":data_dir + name
#     })

# for path in data[:1]:
#     image=Image.open(path["image"])
#     image=np.array(image.convert("RGB")) / 255
#     print("Image:")
#     plt.imshow(image)
#     plt.axis('off')
#     plt.show()
    
#     image=Image.open(path["annotated"])
#     image=np.array(image.convert("RGB")) / 255
#     print("Annotated:")
#     plt.imshow(image)
#     plt.axis('off')
#     plt.show()

In [ ]:
# # use class dict to get ground truth masks by type (key is corresponding english label)
# color_to_class_dict = {tuple(color) : label for label, color in class_dict.items()}
# print(color_to_class_dict)

In [35]:
# def read_batch(data):
#     # Read a random image from the data array of paths
#     entry = data[np.random.randint(len(data))]
#     image = Image.open(entry["image"])
#     image=np.array(image.convert("RGB")) 
#     ann_map = Image.open(entry["annotated"])
#     ann_map=np.array(ann_map.convert("RGB")) 

#     # Note that if any dimension is bigger than 1024px, we must resize it.
#     # For now, images are 720 X 960. No resize needed 

#     # Sort annotation map into list of masks
    
#     ground_truth_object_dict = {}
#     for i in range(ann_map.shape[0]):
#         for j in range(ann_map.shape[1]):
#             color_key = tuple(ann_map[i][j].tolist())

#             color_key = tuple([e / 255 for e in color_key])

#             label = color_to_class_dict[color_key]

#             # track pixel in relevant label
#             if label not in ground_truth_object_dict:
#                 # create boolean mask
#                 ground_truth_object_dict[label] = np.zeros((720, 960), dtype=np.float32)
#             ground_truth_object_dict[label][i][j] = 1

#     masks =  np.array(list(ground_truth_object_dict.values()))
#     points = []

#     for mask in masks:
#         points.append(mask[0, 0])

#     return image,np.array(masks),np.array(points),np.ones([len(masks), 1])


# # read_batch(data=data)

In [36]:
# sam2_checkpoint = "./checkpoints/sam2.1_hiera_small.pt"
# model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"
# sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
# predictor = SAM2ImagePredictor(sam2_model)

In [ ]:
# predictor.model.sam_mask_decoder.train(True) # enable training of mask decoder 
# predictor.model.sam_prompt_encoder.train(True) # enable training of prompt encoder

# optimizer=torch.optim.AdamW(params=predictor.model.parameters(),lr=1e-5,weight_decay=4e-5)
# scaler = torch.cuda.amp.GradScaler("cuda") # set mixed precision - memory efficient

In [ ]:
# for itr in range(100000):
#     with torch.cuda.amp.autocast(True): # cast to mix precision
#             image,mask,input_point, input_label = read_batch(data) # load data batch
#             if mask.shape[0]==0: continue # ignore empty batches
#             predictor.set_image(image) # apply SAM image encoder to the image

#             # Prompt encoding
#             mask_input, unnorm_coords, labels, unnorm_box = predictor._prep_prompts(
#                 input_point, input_label, box=None, mask_logits=None, normalize_coords=True
#             )

#             # Ensure correct shape
#             if unnorm_coords.dim() == 2:  # If shape is (B, 2), add an extra dimension for points
#                 unnorm_coords = unnorm_coords.unsqueeze(1)  # Shape becomes (B, N, 2)

#             if labels.dim() == 2:  # If shape is (B, N), add an extra dimension for labels
#                 labels = labels.unsqueeze(-1)  # Shape becomes (B, N, 1)

#             print("Updated Shapes:", unnorm_coords.shape, labels.shape)  # Debugging

#             # Prompt encoding
#             sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(
#                 points=(unnorm_coords, labels), boxes=None, masks=None
#             )

#             # Mask decoder
#             batched_mode = unnorm_coords.shape[0] > 1  # Multi-mask prediction check
#             high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in predictor._features["high_res_feats"]]

#             low_res_masks, prd_scores, _, _ = predictor.model.sam_mask_decoder(
#                 image_embeddings=predictor._features["image_embed"][-1].unsqueeze(0),
#                 image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),
#                 sparse_prompt_embeddings=sparse_embeddings,
#                 dense_prompt_embeddings=dense_embeddings,
#                 multimask_output=True,
#                 repeat_image=batched_mode,
#                 high_res_features=high_res_features,
#             )

#             # Upscale the masks to the original image resolution
#             prd_masks = predictor._transforms.postprocess_masks(
#                 low_res_masks, predictor._orig_hw[-1]
#             )
#             # Segmentaion Loss caclulation
#             gt_mask = torch.tensor(mask.astype(np.float32)).cuda()
#             prd_mask = torch.sigmoid(prd_masks[:, 0])# Turn logit map to probability map
#             seg_loss = (-gt_mask * torch.log(prd_mask + 0.00001) - (1 - gt_mask) * torch.log((1 - prd_mask) + 0.00001)).mean() # cross entropy loss

#             # Score loss calculation (intersection over union) IOU
#             inter = (gt_mask * (prd_mask > 0.5)).sum(1).sum(1)
#             iou = inter / (gt_mask.sum(1).sum(1) + (prd_mask > 0.5).sum(1).sum(1) - inter)
#             score_loss = torch.abs(prd_scores[:, 0] - iou).mean()
#             loss=seg_loss+score_loss*0.05  # mix losses

#             # apply back propogation
#             predictor.model.zero_grad() # empty gradient
#             scaler.scale(loss).backward()  # Backpropogate
#             scaler.step(optimizer)
#             scaler.update() # Mix precision

#             if itr%1000==0: torch.save(predictor.model.state_dict(), "model.torch")
#             print("Training complete - saved model.")

#             # Display results
#             if itr==0: mean_iou=0
#             mean_iou = mean_iou * 0.99 + 0.01 * np.mean(iou.cpu().detach().numpy())
#             print("step)",itr, "Accuracy(IOU)=",mean_iou)

### Old code using YOLOv2 and SAM2:

In [ ]:
# yolo_model = YOLO('yolov8n.pt')

# # sam2_checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
# sam2_checkpoint = "./checkpoints/sam2.1_hiera_small.pt"
# model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
# sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
# predictor = SAM2ImagePredictor(sam2_model)

In [ ]:
# def yoloSAM2_segment_and_show(img_path, show_base_image=False):
#     results = yolo_model.predict(source=img_path, conf=0.25)
#     names = results[0].names

#     for result in results: # Usually results is of length 1 - can change
#         boxes = result.boxes
#         bbox = boxes.xyxy.tolist() # bounding boxes AKA areas where YOLO found an object
        
#         # print(boxes.cls)
#         # print(result.names)

#         image=Image.open(img_path)
#         image=np.array(image.convert("RGB"))

#         if show_base_image:
#             plt.figure(figsize=(20, 20))
#             plt.imshow(image)
#             plt.axis('off')
#             plt.show()

#         object_mask_dict, list_of_masks = SAM2_segment_box(image=image, predictor=predictor, bbox=bbox, labels = [names[index] for index in boxes.cls.tolist()])
#         print(f"Found {len(list_of_masks)} objects in image.") 
    
#     return object_mask_dict, list_of_masks

In [ ]:
# def intersection_over_union(predicted:np.array, ground:np.array):
#     # calcualte intersection over union for two masks.
#     # return number of pixels (int).
#     intersection = 0
#     union = 0
#     height = len(ground)
#     length = len(ground[0])
#     for i in range(height):
#         for j in range(length):
#             if predicted[i][j] != 0 and ground[i][j] != 0:
#                 intersection += 1
#             if predicted[i][j] != 0 or ground[i][j] != 0:
#                 union += 1
                
#     return intersection / union if union != 0 else len(ground[np.nonzero(ground)].ravel())

In [22]:
# def calc_loss(predicted, ground):
#     """
#         Given a list of masks, where each mask is a numpy array representing the entire image 
#         but with logical 1's denoting pixels that are members of the mask,
#         calculate and return the loss.
#     """
#     # For-loop over labels
#     for label in ground:
#         if label in predicted:
#             # Case 1 : SAM2 did not find label (penalize for it)
#             pass
#         else:
#             # Case 2 : SAM2 did find label
#             pass

#     # print(list(predicted.keys()), list(ground.keys()))


In [ ]:
# yolo_model = YOLO('yolov8n.pt')

# # sam2_checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
# sam2_checkpoint = "./checkpoints/sam2.1_hiera_small.pt"
# model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
# sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
# predictor = SAM2ImagePredictor(sam2_model)

In [ ]:
# def yoloSAM2_segment_and_show(img_path, show_base_image=False):
#     results = yolo_model.predict(source=img_path, conf=0.25)
#     names = results[0].names

#     for result in results: # Usually results is of length 1 - can change
#         boxes = result.boxes
#         bbox = boxes.xyxy.tolist() # bounding boxes AKA areas where YOLO found an object
        
#         # print(boxes.cls)
#         # print(result.names)

#         image=Image.open(img_path)
#         image=np.array(image.convert("RGB"))

#         if show_base_image:
#             plt.figure(figsize=(20, 20))
#             plt.imshow(image)
#             plt.axis('off')
#             plt.show()

#         object_mask_dict, list_of_masks = SAM2_segment_box(image=image, predictor=predictor, bbox=bbox, labels = [names[index] for index in boxes.cls.tolist()])
#         print(f"Found {len(list_of_masks)} objects in image.") 
    
#     return object_mask_dict, list_of_masks

In [ ]:
# def intersection_over_union(predicted:np.array, ground:np.array):
#     # calcualte intersection over union for two masks.
#     # return number of pixels (int).
#     intersection = 0
#     union = 0
#     height = len(ground)
#     length = len(ground[0])
#     for i in range(height):
#         for j in range(length):
#             if predicted[i][j] != 0 and ground[i][j] != 0:
#                 intersection += 1
#             if predicted[i][j] != 0 or ground[i][j] != 0:
#                 union += 1
                
#     return intersection / union if union != 0 else len(ground[np.nonzero(ground)].ravel())

In [ ]:
# # convert image from path to numpy array
# img_path = "../download_dataset/train/0001TP_009420.png"
# validation_path = "../download_dataset/train_labels/0001TP_009420_L.png"

# image=Image.open(validation_path)
# image=np.array(image.convert("RGB")) / 255
# print("Ground truth:")
# plt.imshow(image)
# plt.axis('off')
# plt.show()

# # print(tuple(image[0][0].tolist()))

# ground_truth_object_dict = {}
# for i in range(image.shape[0]):
#     for j in range(image.shape[1]):
#         color_key = tuple(image[i][j].tolist())
#         label = color_to_class_dict[color_key]

#         # track pixel in relevant label
#         if label not in ground_truth_object_dict:
#             # create boolean mask
#             ground_truth_object_dict[label] = np.zeros((720, 960), dtype=np.float32)
#         ground_truth_object_dict[label][i][j] = 1

# # print(list(ground_truth_object_dict.keys()))
# # print(ground_truth_object_dict['car'])

# # get SAM2 attempt and calculate loss
# sam2_mask_dict, list_of_masks = yoloSAM2_segment_and_show(img_path=img_path)

# # rename labels with camvid labels, if they differ
# sam2_mask_dict = { label_map[key] if key in label_map else key:value for key, value in sam2_mask_dict.items() }
# # for key in sam2_mask_dict:
# #     if key in label_map:
# #         sam2_mask_dict[label_map[key]] = sam2_mask_dict.pop(key)


# print("SAM2 prediction found:", list(sam2_mask_dict.keys()))
# print("Ground truth has:", list(ground_truth_object_dict.keys()))

# # print(calc_loss(predicted=sam2_mask_dict, ground=ground_truth_object_dict))

# SAM2, then YOLO
SAM2 segment for masks, YOLO label masks